---
## 1. Ensure src/ is importable + build the article index

# Legal Explainer — Pipeline Smoke Test

Exercises every piece of the agent system end-to-end:

1. Build the statute article index from `EgyptianLaw.pdf`
2. Sanity-check each tool in isolation (glossary, statute, RAG, web search)
3. Sanity-check the router on each complexity bucket
4. Run the orchestrator on one query per path (simple / medium / complex)
5. Run the baseline (Researcher → Explainer) on the same query for comparison

**Prereqs:**
- Ollama running locally with `qwen3-embedding:0.6b` pulled
- `ingest_rag.ipynb` has finished (so `rag_storage_egyptian_law/` exists)
- `.env` populated with `ANTHROPIC_AUTH_TOKEN` + `ANTHROPIC_BASE_URL`
- Python env has: `claude-agent-sdk`, `lightrag-hku`, `pymupdf`, `httpx`, `duckduckgo-search`

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path("/Volumes/Shared/NileUni/GenAI/LegalPolicy_LLM")
SRC = PROJECT_ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from legal_explainer.agents.tools.statute import build_index, ARTICLES_INDEX_PATH

if not ARTICLES_INDEX_PATH.exists():
    print("Article index not found — building from EgyptianLaw.pdf ...")
    idx = build_index()
    print(f"Built {len(idx)} articles")
else:
    import json
    idx = json.loads(ARTICLES_INDEX_PATH.read_text())
    print(f"Loaded existing index: {len(idx)} articles")

Loaded existing index: 2164 articles


---
## 2. Test each tool standalone

In [2]:
from legal_explainer.agents.tools.glossary import lookup_definition

print("--- Glossary: English ---")
print(lookup_definition("indemnity")["en"]["definition"])

print("\n--- Glossary: Arabic alias resolves to same entry ---")
print(lookup_definition("تعويض")["canonical_id"])

print("\n--- Glossary: miss ---")
r = lookup_definition("copyright")
print({"found": r["found"], "available_count": len(r["available_terms"])})

--- Glossary: English ---
Monetary compensation awarded to a party who has suffered loss or injury due to another's act or omission.

--- Glossary: Arabic alias resolves to same entry ---
damages

--- Glossary: miss ---
{'found': False, 'available_count': 15}


In [3]:
from legal_explainer.agents.tools.statute import lookup_article

print("--- Statute: English reference ---")
r = lookup_article("Article 89")
print({"found": r["found"], "text_preview": r.get("text", "")[:300]})

print("\n--- Statute: Arabic reference ---")
r = lookup_article("المادة 936")
print({"found": r["found"], "text_preview": r.get("text", "")[:300]})

print("\n--- Statute: unparseable ---")
print(lookup_article("the third paragraph"))

--- Statute: English reference ---
{'found': True, 'text_preview': 'A contract is created, subject to any special formalities \nthat may be required by law for its conclusion, from the \nmoment that two persons have exchanged two \nconcordant intentions.\n ('}

--- Statute: Arabic reference ---
{'found': True, 'text_preview': 'The right of preemption belongs: \na- to the bare owner, in the case of a sale of all or part of \nthe usufruct attached to a bare property; \n\n\n )\n جـ) لصاحب حق االنتفاع إذا بيعت كل الرقبة المالبسة لهذا الحق أو\nبعضها .\n \n )\n د) لمالك الرقبة فى الحكر إذا بيع حق الحكر ، وللمستحكر إذا بيعت\nالرقبة .\n \n )ه'}

--- Statute: unparseable ---
{'found': False, 'queried_reference': 'the third paragraph', 'reason': 'could_not_parse', 'message': "Could not extract an article number from the reference. Try formats like 'Article 89' or 'المادة 89'."}


In [ ]:
# RAG search — single-arg call: query in, context out.
# `search_legal_context` is the pure-Python entrypoint (the @tool-wrapped
# `search_legal_documents` calls into this same function).
# Sensible defaults baked in: mode=hybrid, top_k=5, enable_rerank=False.
# Zero LLM calls on LightRAG's side (pre-extracted keywords + aquery_data).
from legal_explainer.agents.tools.rag_search import search_legal_context

payload = await search_legal_context("What does Egyptian law say about gifts and revocation?")

print(f"ok: {payload['ok']}\n")
if payload["ok"]:
    print(f"entities:      {len(payload['entities'])}")
    print(f"relationships: {len(payload['relationships'])}")
    print(f"chunks:        {len(payload['chunks'])}")

    print("\nFirst 3 entities:")
    for e in payload["entities"][:3]:
        print(f"  - {e['name']} ({e['type']})")
        print(f"    {(e.get('description') or '')[:160]}")

    print("\nFirst chunk preview:")
    print((payload["chunks"][0]["content"] or "")[:600])
else:
    print(f"message: {payload.get('message')}")

---
## 3. Test the router (rules first, LLM fallback)

In [ ]:
from legal_explainer.agents.tools.router import classify_complexity

test_queries = [
    "What is force majeure?",                                  # simple — glossary hit
    "ما معنى التعويض؟",                                          # simple — Arabic glossary
    "Walk me through Article 713 of the Egyptian Civil Code.", # medium — article ref
    "Compare Article 660 and Article 713 in terms of duties.", # complex — comparison
    "How does Egyptian law approach co-ownership disputes?",   # LLM fallback expected
]

for q in test_queries:
    d = await classify_complexity(q)
    print(f"  [{d.complexity:<8}] llm={d.used_llm}  rule={d.rule_matched!r}  | {q!r}")

---
## 4. Orchestrator — one query per path

Each call returns an `OrchestratorResult` with the answer + which path ran + cost + duration. Every step is also written to `reports/agent_traces/run_YYYYMMDD.jsonl`.

In [ ]:
from legal_explainer.agents.orchestrator import run_orchestrated

# Simple path — glossary tool, no subagent
r = await run_orchestrated("What is force majeure?")
print(f"path={r.path}  complexity={r.complexity}  duration={r.duration_ms}ms  cost=${r.total_cost_usd:.4f}")
print("\n--- ANSWER ---")
print(r.answer)

In [ ]:
# Medium path — article reference → Researcher + Explainer
r = await run_orchestrated("Walk me through Article 713 of the Egyptian Civil Code.")
print(f"path={r.path}  complexity={r.complexity}  duration={r.duration_ms}ms  cost=${r.total_cost_usd:.4f}")
print("\n--- ANSWER ---")
print(r.answer)

In [ ]:
# Complex path — Comparator + Explainer
r = await run_orchestrated(
    "Compare Article 660 and Article 713 in terms of who is bound and what is owed."
)
print(f"path={r.path}  complexity={r.complexity}  duration={r.duration_ms}ms  cost=${r.total_cost_usd:.4f}")
print("\n--- ANSWER ---")
print(r.answer)

print("\n--- TRACE STEPS ---")
for step in r.steps:
    print(f"  [{step['step']}] {step}")

In [ ]:
# Refusal path — should never hit any subagent
r = await run_orchestrated("I was arrested last night, should I sign this contract?")
print(f"path={r.path}  safety={r.safety_verdict}  cost=${r.total_cost_usd:.4f}")
print("\n--- ANSWER ---")
print(r.answer)

---
## 5. Baseline pipeline (Epic 7 task 7.1)

Always Researcher → Explainer, no routing. Same query as the medium-path test above so you can compare side-by-side.

In [ ]:
from legal_explainer.agents.orchestrator import run_baseline

r_baseline = await run_baseline("Walk me through Article 713 of the Egyptian Civil Code.")
print(f"path={r_baseline.path}  duration={r_baseline.duration_ms}ms  cost=${r_baseline.total_cost_usd:.4f}")
print("\n--- ANSWER ---")
print(r_baseline.answer)

---
## 6. Tool-use ablation (Epic 6 task 6.4)

Same definition-style query, with and without the glossary tool. The first should hit the curated entry (deterministic, citable); the second falls through to RAG/general knowledge and produces a less precise answer.

In [ ]:
# With glossary tool (default orchestrator behaviour — simple path)
r_with = await run_orchestrated("What is force majeure?")
print("=== WITH glossary ===")
print(r_with.answer)
print(f"\npath={r_with.path}, cost=${r_with.total_cost_usd:.4f}")

# Without glossary — force the baseline path which has no router/glossary shortcut
r_without = await run_baseline("What is force majeure?")
print("\n=== WITHOUT glossary (baseline pipeline) ===")
print(r_without.answer)
print(f"\npath={r_without.path}, cost=${r_without.total_cost_usd:.4f}")

---
## 7. Mini eval (5 cases from `qa_pairs_raft_val.jsonl`)

Run the orchestrator on a tiny sample to verify the full eval harness works. For a real run, use the CLI:

```bash
python -m legal_explainer.agents.eval.run_eval --system orchestrated --n 21
python -m legal_explainer.agents.eval.run_eval --system baseline    --n 21
python -m legal_explainer.agents.eval.compare_baseline \
    --baseline reports/agent_eval/predictions_baseline.json \
    --orchestrated reports/agent_eval/predictions_orchestrated.json
```

In [ ]:
from legal_explainer.agents.eval.run_eval import _load_cases, _retrieval_hit, DEFAULT_INPUT

cases = _load_cases(DEFAULT_INPUT, n=5)
for i, c in enumerate(cases, 1):
    print(f"\n[{i}] {c['language']} {c['kind']}  article={c['article_key']}")
    print(f"   Q: {c['user_query'][:120]}...")
    r = await run_orchestrated(c["user_query"])
    hit = _retrieval_hit(r.answer, c["article_key"])
    print(f"   path={r.path}  hit={hit}  duration={r.duration_ms}ms")
    print(f"   A: {r.answer[:200]}...")